In [0]:
# Dependencies
%pip install databricks-zerobus-ingest-sdk

import logging
import datetime, random, time
from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import TableProperties, StreamConfigurationOptions, RecordType
from Lab3.sensor_stream import make_event, SITES
from pyspark.sql import functions as F, Window

In [0]:
# Configuration

dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("bronze_schema", "bronze", ["bronze", "gabrielajaniszews786_bronze"], "Bronze schema")
dbutils.secrets.get(scope = "entsoe", key = "zerobus-client-id-gabriela")
CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
WORKSPACE_URL = "https://adb-7405615123305702.2.azuredatabricks.net"
SERVER_ENDPOINT = "https://7405615123305702.zerobus.eastus.azuredatabricks.net"
CLIENT_ID = dbutils.secrets.get(scope = "entsoe", key = "zerobus-client-id-gabriela")
CLIENT_SECRET = dbutils.secrets.get(scope = "entsoe", key = "zerobus-client-secret-gabriela")

ROUNDS = 20
SLEEP_S = 3

In [0]:
# Creating a new table for streaming
spark.sql(f"""DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.zerobus_bronze""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.zerobus_bronze (
    event_id STRING,
    schema_version INT,
    site_id STRING,
    site_name STRING,
    country STRING,
    bidding_zone STRING,
    timestamp_utc STRING,
    consumption_kwh DOUBLE,
    avg_power_kw DOUBLE,
    pue DOUBLE,
    reading_interval_s INTEGER
)
USING DELTA
""")


In [0]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
)

In [0]:
print(CLIENT_ID[:8], CLIENT_ID[-4:])
print(len(CLIENT_SECRET), CLIENT_SECRET[:4])

In [0]:
TABLE_NAME = f"{CATALOG}.{BRONZE_SCHEMA}.zerobus_bronze"

# Initialize SDK
sdk = ZerobusSdk(SERVER_ENDPOINT, WORKSPACE_URL)

# Configure table properties
table_properties = TableProperties(
    table_name=TABLE_NAME,
)

# Create stream
stream = sdk.create_stream(
    CLIENT_ID,
    CLIENT_SECRET,
    table_properties,
    options=StreamConfigurationOptions(record_type=RecordType.JSON))


In [0]:
# Sending the data
sent = 0

try:
    for _ in range(ROUNDS):
        records = []
        ts   = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

        for site in SITES:
            avg_power = random.uniform(700.0, 800.0)
            event = make_event(
                site, 
                timestamp_utc=ts,
                consumption_kwh = round(avg_power * random.uniform(0.85, 1.25), 2),
                avg_power_kw=avg_power,
                pue = random.uniform(1.2, 1.4),
            )
            records.append(event.to_dict())
            sent +=1
            
        stream.ingest_records_offset(records)
        time.sleep(SLEEP_S)
        print(f"Sent {sent} events from {len(SITES)} sites")
        stream.flush()
finally:
    stream.close()